## Surrogate vs. Thermochemical library
### **Computational speedup**
Import libraries

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm import tqdm

In [2]:
sys.path.append(os.path.abspath('./functions'))

import stats
import plotting as pltt

Define inputs

In [3]:
# 'csv' header
header = ["t", "mean", "std"]
# Paths to timings
prefix = "/home/zanardi/Codes/ML/PyCOMET/run/meno/speedup/"
# > ODE solver
k = "ode"
timingsf = {k: prefix + f"{k}/data/timings.csv"}
# > Surrogate
k = "sur"
models = ("sp", "mp")
devices = ("cpu", "gpu")
timingsf[k] = {}
for model in models:
  timingsf[k][model] = {}
  for device in devices:
    timingsf[k][model][device] = prefix + f"{k}/data/timings_{model}_pade_{device}.csv"

In [ ]:
path_to_saving = os.path.abspath("./output/comp_time/")
os.makedirs(path_to_saving, exist_ok=True)

Read timings

In [4]:
def read_timings(filename):
  timings = pd.read_csv(filename)
  timings.columns = header
  return timings

In [5]:
timings = tf.nest.map_structure(read_timings, timingsf)

Compute speedup

In [6]:
def compute_speedup(tode, tsur):
  # Extract mean and std
  values = {}
  for k in ("mean", "std"):
    values[k] = np.vstack([df[k].to_numpy().squeeze() for df in (tode, tsur)])
  # Compute speedup
  rmean, rstd = stats.ratio_normal_approx(values["mean"], values["std"])
  # Compose data
  time = tode["t"].to_numpy().squeeze()
  data = np.vstack([time, rmean, rstd]).T
  return pd.DataFrame(data=data, columns=header)

In [7]:
speedup = tf.nest.map_structure(
  lambda tsur: compute_speedup(timings["ode"], tsur), timings["sur"]
)

Plot

In [8]:
x = timings["ode"]["t"].to_numpy().squeeze()

> Timings

In [10]:
for model in models:
  values = {}
  for k in ("mean", "std"):
    values[k] = [timings["ode"][k].to_numpy().squeeze()]
    for d in devices:
      y = timings["sur"][model][d][k].to_numpy().squeeze()
      values[k].append(y)
    values[k] = np.vstack(values[k])
  pltt.timings(
    x,
    mean=values["mean"],
    std=values["std"],
    var_names=["1 CPU ODE"] + [f"1 {d.upper()} SUR" for d in devices],
    labels=["Physical time [s]", "Integration time [s]"],
    scales=['log', 'log'],
    figname=path_to_saving + f"/timings_{model}",
    save=True,
    show=False,
    extensions=('png',)
  )

> Speedup

In [12]:
for model in models:
  values = {}
  for k in ("mean", "std"):
    values[k] = []
    for d in devices:
      y = speedup[model][d][k].to_numpy().squeeze()
      values[k].append(y)
    values[k] = np.vstack(values[k])
  pltt.timings(
    x,
    mean=values["mean"],
    std=values["std"],
    var_names=[f"1 CPU ODE vs. 1 {d.upper()} SUR" for d in devices],
    labels=["Physical time [s]", "Speedup"],
    scales=['log', 'log'],
    figname=path_to_saving + f"/speedup_{model}",
    save=True,
    show=False,
    extensions=('png',)
  )